In [1]:
import fitz
import pandas as pd
import io
import glob
import re

result_list = {'title': [], 'namecompany': [], 'description': [], 'rating': [], 'field': [], 'date': [], 'textpub': []}

In [2]:
all_pdf = glob.glob(r"PDF\*.pdf")

In [3]:
all_pdf

['PDF\\AGI уже стучится в дверь человечества. А мы всё ещё думаем, что он станет нашим лучшим другом _ Хабр.pdf',
 'PDF\\ASR на CPU. Как выбрать бэкенд, настроить Triton и не потерять в точности _ Хабр.pdf',
 'PDF\\Claude Code теперь требует паспорт, а Codex умеет работать с (почти) любыми приложениями _ Хабр.pdf',
 'PDF\\Microsoft заверила, что Windows 11 прекрасно обходится без стороннего антивируса _ Хабр.pdf',
 'PDF\\Opus 4.7 — худший релиз в истории Anthropic_ _ Хабр.pdf',
 'PDF\\Вайбкодинг — это плохо_ _ Хабр.pdf',
 'PDF\\КРОК открывает набор на Летнюю ИТ-школу 2026 _ Хабр.pdf',
 'PDF\\Наплыв музыки, сгенерированной ИИ, меняет подход стриминговых платформ к обработке новых загрузок _ Хабр.pdf',
 'PDF\\Сэм Альтман критикует кибермодель Mythos от Anthropic _ Хабр.pdf',
 'PDF\\Топ 12 сервисов и нейросетей для генерации изображения 2026_ подборка AI фотографов _ Хабр.pdf']

In [4]:
def extr(pdf):
    doc = fitz.open(pdf)
    text = []
    for current_page in range(len(doc)):
        page_text = doc.load_page(current_page).get_text("text")
        for i in page_text.split('\n'):
            i = re.sub(r'\s+', ' ', i).strip()
            if i:
                text.append(i)
    return doc.name[4:].split(' _ Хабр.pdf')[0], '\n'.join(text)

In [5]:
def pars_pdf(text):
    x = text.split('\n')
    comp = x[2]
    desc = x[3]
    rate = next((s for s in x if "Рейтинг" in s)).split('Рейтинг')[0].strip()
    fields = ', '.join(x[x.index(re.findall(r"\d+(?:\.\d+)?[KК]", text)[1]) + 1].replace('*','').split(',')[:-1])
    return comp, desc, rate, fields

In [6]:
for doc in all_pdf:
    title, text = extr(doc)
    c,d,r,f = pars_pdf(text)
    date_str = fitz.open(doc).metadata['creationDate']
    result_list['title'].append(title)
    result_list['namecompany'].append(c)
    result_list['description'].append(d)
    result_list['rating'].append(r)
    result_list['field'].append(f)
    result_list['date'].append(f"{date_str[2:6]}-{date_str[6:8]}-{date_str[8:10]}")
    result_list['textpub'].append(text)

In [7]:
file_name = 'All_pdf_stats.csv'
df = pd.DataFrame(data=result_list)
df.to_csv(file_name)
df.head()

,title,namecompany,description,rating,field,date,textpub
0,AGI уже стучится в дверь человечества. А мы вс...,Timeweb Cloud,То самое облако,"1 560,8","Блог компании Timeweb Cloud, Искусственный ин...",2026-04-22,256K+\nОхват за 30 дней\nTimeweb Cloud\nТо сам...
1,"ASR на CPU. Как выбрать бэкенд, настроить Trit...",MWS AI,Создаем решения будущего уже сегодня,"78,94","Блог компании MWS AI, Блог компании МТС, Маш...",2026-04-22,8K+\nОхват за 30 дней\nMWS AI\nСоздаем решения...
2,"Claude Code теперь требует паспорт, а Codex ум...",Haulmont,Корпоративные системы и инструменты разработчика,"201,84","Блог компании Haulmont, Программирование",2026-04-22,64K+\nОхват за 30 дней\nHaulmont\nКорпоративны...
3,"Microsoft заверила, что Windows 11 прекрасно о...",BotHub,Bothub — экосистема AI инструментов,"383,71","Блог компании BotHub, Операционные системы, ...",2026-04-22,512K+\nОхват за 30 дней\nBotHub\nBothub — экос...
4,Opus 4.7 — худший релиз в истории Anthropic_,BotHub,Bothub — экосистема AI инструментов,"383,71","Блог компании BotHub, Искусственный интеллект...",2026-04-22,512K+\nОхват за 30 дней\nBotHub\nBothub — экос...
